# Getting started — Text-to-Speech Model

Chatterbox Turbo on SageMaker. Send JSON to `/invocations` and get raw `audio/wav` bytes.

This notebook deploys the Marketplace model package to a real-time endpoint, synthesizes a short phrase, saves the WAV, then deletes the endpoint. Real-time endpoints are on-demand (not spot) and bill until deleted.

**Before you run this:**
1. Subscribe on AWS Marketplace (**Continue to Subscribe** → **Continue to Configuration**).
2. Paste the **Model Package ARN** from that page into the next cell. That ARN is AWS's copy of the package (unique per region), not the seller ARN.
3. Run from SageMaker Studio / a notebook instance (execution role auto-detected), or set `ROLE_ARN` yourself. The role needs `sagemaker:CreateModel`, `sagemaker:CreateEndpoint`, and `sagemaker-runtime:InvokeEndpoint`.
4. Use a GPU instance type (`ml.g4dn.xlarge` recommended). First startup can take several minutes while weights load onto the GPU.

Product page: https://www.sigmodata.com/products/text-to-speech-model  
Download this notebook: https://www.sigmodata.com/products/text-to-speech-model/getting-started.ipynb

In [ ]:
import json
import time
from pathlib import Path

import boto3
from botocore.config import Config

# Product ARN from AWS Marketplace → Continue to Configuration.
MODEL_PACKAGE_ARN = ""
assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), (
    "Paste the Model Package ARN from Marketplace Continue to Configuration"
)
REGION = MODEL_PACKAGE_ARN.split(":")[3]
INSTANCE_TYPE = "ml.g4dn.xlarge"  # GPU recommended; CPU works but is slow
TEXT = "Hello from the Text-to-Speech Model on SageMaker."
OUT_WAV = Path("sagemaker-tts.wav")

session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
runtime = session.client(
    "sagemaker-runtime",
    config=Config(read_timeout=600, retries={"max_attempts": 0}),
)
sts = session.client("sts")
account = sts.get_caller_identity()["Account"]
ROLE_ARN = (
    f"arn:aws:iam::{account}:role/service-role/AmazonSageMaker-ExecutionRole"
)
print("account", account)
print("role   ", ROLE_ARN)
print("arn    ", MODEL_PACKAGE_ARN)

## 1. Deploy a real-time endpoint

SageMaker starts the container with a literal `serve` argument: only `GET /ping`, `POST /invocations`, and `GET /execution-parameters`. No UI, no `/synthesis`, no auth.

Startup can take several minutes while Chatterbox Turbo loads onto the GPU. `ContainerStartupHealthCheckTimeoutInSeconds=900` covers that.

In [ ]:
name = f"sigmodata-tts-{int(time.time())}"
print("endpoint", name)

sm.create_model(
    ModelName=name,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InitialInstanceCount": 1,
        "InstanceType": INSTANCE_TYPE,
        "ContainerStartupHealthCheckTimeoutInSeconds": 900,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print("Waiting for InService (model load can take several minutes)...")
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 60}
)
print(sm.describe_endpoint(EndpointName=name)["EndpointStatus"])

## 2. Synthesize speech

POST JSON with `input` (or `text`), optional `voice`, `language`, and `response_format` (`wav` or `aiff`). Set `Accept` to `audio/wav`. The response is **raw audio bytes**, not JSON. Real-time payload cap is **6 MiB**; use batch transform for longer inputs.

In [ ]:
payload = json.dumps({
    "input": TEXT,
    "voice": "default",
    "response_format": "wav",
}).encode()

t0 = time.time()
resp = runtime.invoke_endpoint(
    EndpointName=name,
    ContentType="application/json",
    Accept="audio/wav",
    Body=payload,
)
wav = resp["Body"].read()
elapsed = time.time() - t0
assert wav[:4] == b"RIFF", f"expected RIFF WAV, got {wav[:16]!r}"
OUT_WAV.write_bytes(wav)
print(f"OK ({elapsed:.1f}s, {len(wav)} bytes) -> {OUT_WAV.resolve()}")

## 3. Clean up the endpoint

Real-time endpoints keep billing until deleted. Always run this when finished.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.get_waiter("endpoint_deleted").wait(
    EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 40}
)
sm.delete_endpoint_config(EndpointConfigName=name)
sm.delete_model(ModelName=name)
print("deleted", name)

## Batch transform (optional)

Put one JSON object per line in S3 and run a transform job. This model is `SINGLE_RECORD` / `MaxPayloadInMB=6` (see `GET /execution-parameters`). Example input line:

`{"input": "Hello", "voice": "default", "response_format": "wav"}`

In [ ]:
# bucket = "<your-bucket>"
# prefix = "tts-batch-example"
# s3 = session.client("s3")
# line = json.dumps({"input": TEXT, "voice": "default", "response_format": "wav"})
# key = f"{prefix}/input.jsonl"
# s3.put_object(Bucket=bucket, Key=key, Body=(line + "\n").encode())
# input_s3 = f"s3://{bucket}/{prefix}/"
# output_s3 = f"s3://{bucket}/{prefix}/out/"
# job = f"tts-batch-{int(time.time())}"
# sm.create_transform_job(
#     TransformJobName=job,
#     ModelName=name,
#     MaxConcurrentTransforms=1,
#     MaxPayloadInMB=6,
#     BatchStrategy="SingleRecord",
#     TransformInput={
#         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": input_s3}},
#         "ContentType": "application/json",
#         "SplitType": "Line",
#     },
#     TransformOutput={"S3OutputPath": output_s3, "Accept": "audio/wav"},
#     TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1},
# )

## Next steps

- Product page: https://www.sigmodata.com/products/text-to-speech-model
- Self-hosted HTTPS UI, batch folders, zero-shot cloning: [Text-to-Speech Server](https://www.sigmodata.com/products/text-to-speech-server)
- Support: support@sigmodata.com